# 07. Derivatives Hedging — 파생상품으로 매입 가격 위험 줄이기

이 노트북은 **교육적 분석**입니다. 패션 산업에서 거래소 상장 선물·옵션이 광범위하게 존재하지는 않습니다 (오일·금속·곡물처럼). 그러나 다음의 경우 **OTC 파생, 사내 헷징 정책, 공급사와의 forward 계약**을 설계할 때 동일한 수학이 쓰입니다.

- 환율 변동에 노출된 수입 매입가 (KRW/USD forward, FX option)
- 유가에 연동된 운송·물류 비용 (oil futures)
- 원자재 비용 (cotton futures, 의류 산업의 일부 원자재는 거래소 상장)

이 노트북의 목표는 "수치 한 줄로 가격 헷징의 손익 구조를 시각화"하는 것입니다. 실제 의사결정에는 회계·법무·트레이딩 데스크 검토가 필요합니다.

## 다루는 도구

1. **Forward 계약** — 미래 N개월 후 정해진 가격 K로 **매입 약속**. 비용 0(증거금 별도). 가격이 K보다 오르면 이익.
2. **Futures** — 거래소 상장 forward. 일일정산(mark-to-market)이 있어 현금 흐름 부담.
3. **Call option** — 만기에 K로 **매입할 권리**(의무 X). 프리미엄 지급. 가격이 K+프리미엄 이상으로 오르면 행사.
4. **Put option** — 매입 입장에서는 거의 쓰지 않음. 판매자(공급사) 헷징용.
5. **Collar** — call 매수 + put 매도로 프리미엄 상쇄. 상승 위험 헷지 + 하락 이익 일부 포기.

## 사용 패키지

- `numpy`, `pandas`, `matplotlib`
- `scipy.stats.norm` — Black-Scholes 정규분포 함수

## ⚠️ 주의

- 이 노트북은 **교육 목적**입니다. 투자·헷징 권유가 아닙니다.
- 실제 헷징 설계는 회계 처리(KIFRS 9), 신용 위험, 베이시스 위험 평가가 필요합니다.
- 모든 모델은 가정에 의존합니다 — Black-Scholes는 변동성 일정·연속 거래·무차익 등을 가정.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')

## 1. 가격 시계열에서 변동성 추정

Black-Scholes에 들어가는 핵심 입력은 **연환산 변동성 σ**. 월별 데이터 → 월별 로그수익률의 표준편차 × √12.

In [ ]:
df = pd.read_csv(DATA_DIR / 'sample_purchases.csv', parse_dates=['date']).sort_values('date')
TARGET = 'SKU-A001-COTTON'
S = (df[df['product_id'] == TARGET]
     .set_index('date')['unit_cost_krw']
     .asfreq('MS').interpolate())

log_ret = np.log(S / S.shift(1)).dropna()
sigma_monthly = log_ret.std()
sigma_annual = sigma_monthly * np.sqrt(12)
S0 = float(S.iloc[-1])
print(f'스팟 가격 S0 = {S0:,.0f} KRW')
print(f'월별 로그수익률 표준편차 = {sigma_monthly:.4f}')
print(f'연환산 변동성 σ = {sigma_annual:.3f}  (= {sigma_annual*100:.1f}%)')

## 2. Forward 가격 — 무차익 가격 (간단형)

보유 비용(c) 고려 안 하면:

$$F = S_0 \cdot e^{r T}$$

$r$ = 무위험 이자율 (한국 국채 단기 금리 등), $T$ = 만기까지 연 단위.

공급사와의 직접 forward 계약은 **시장 forward 가격을 기준선**으로 협상합니다. 이론값 대비 공급사 제시가가 비싼지 싼지가 협상 데이터.

In [ ]:
r = 0.035   # 무위험 이자율 (가정, 본인 환경에 맞게 조정)
T_months_list = [3, 6, 12]
fwd_table = pd.DataFrame({
    'months': T_months_list,
    'T (yr)': [m/12 for m in T_months_list],
    'forward_price': [round(S0 * np.exp(r * m/12), 0) for m in T_months_list],
    'vs_spot_pct': [round((np.exp(r * m/12) - 1) * 100, 2) for m in T_months_list],
})
fwd_table

## 3. Black-Scholes Call/Put 가격

유럽형 call/put 옵션의 이론가:

$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

$$C = S_0 N(d_1) - K e^{-rT} N(d_2)$$
$$P = K e^{-rT} N(-d_2) - S_0 N(-d_1)$$

In [ ]:
def bs_call(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0:
        return max(S - K, 0.0)
    d1 = (np.log(S/K) + (r + sigma**2/2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2)

def bs_put(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0:
        return max(K - S, 0.0)
    d1 = (np.log(S/K) + (r + sigma**2/2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r*T) * norm.cdf(-d2) - S * norm.cdf(-d1)

T = 6/12   # 만기 6개월
strikes = [int(S0 * k) for k in [0.95, 1.00, 1.05, 1.10]]
rows = []
for K in strikes:
    c = bs_call(S0, K, T, r, sigma_annual)
    p = bs_put(S0, K, T, r, sigma_annual)
    rows.append({'strike K': K, 'moneyness': round(K/S0, 3),
                 'call_premium': round(c, 0), 'call_pct': round(c/S0*100, 2),
                 'put_premium': round(p, 0), 'put_pct': round(p/S0*100, 2)})
opt_table = pd.DataFrame(rows)
opt_table

### 해석

- 행사가(K) = 110% 콜 옵션 = "6개월 뒤 가격이 +10% 올라도 지금 가격의 +10%까지만 매입가 노출"
- `call_pct` = 콜 옵션 프리미엄을 스팟 가격 대비 %로 본 것. 이 값이 "위험 보험료"입니다.
- 변동성 σ가 낮을수록 콜 프리미엄 ↓ (안정적 가격은 보험료 쌈).

## 4. Payoff Diagram — 만기 시 시나리오별 손익

구매 입장에서, 만기 가격 $S_T$에 따라 각 전략의 **순매입 비용**이 어떻게 달라지나.

In [ ]:
K = int(S0 * 1.05)    # 5% out-of-the-money call
C = bs_call(S0, K, T, r, sigma_annual)
F = S0 * np.exp(r * T)

S_T = np.linspace(S0 * 0.7, S0 * 1.4, 200)

spot_cost = S_T                                    # 헷지 안 함: 만기 스팟에 매입
fwd_cost = np.full_like(S_T, F)                    # forward: 무조건 F로 매입
call_cost = np.minimum(S_T, K) + C * np.exp(r*T)   # call 보유: max는 K, 프리미엄(미래가치)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(S_T, spot_cost, label='Unhedged (spot)', color='steelblue')
ax.plot(S_T, fwd_cost, label=f'Forward @ {F:,.0f}', color='orange', linestyle='--')
ax.plot(S_T, call_cost, label=f'Call @ K={K:,.0f}, premium={C:,.0f}', color='green')
ax.axvline(S0, color='black', linewidth=0.5, linestyle=':')
ax.text(S0, ax.get_ylim()[1] * 0.95, ' S0', fontsize=9)
ax.set_xlabel('만기 시 스팟 가격 S_T (KRW)')
ax.set_ylabel('단위당 순매입 비용 (KRW)')
ax.set_title(f'{TARGET} — 6개월 만기 헷징 전략 비교')
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(DATA_DIR / f'payoff_{TARGET}.png', dpi=120)
plt.show()

### 읽는 법

- 가로축 = 만기에 **실제로** 도래할 가격(미지)
- 세로축 = 그 시점 1단위를 매입하는 데 드는 **총 비용** (옵션 프리미엄 포함)
- 파란선(헷지 없음): 가격 따라 비용도 그대로 — 상승 무방어
- 주황선(forward): 가격에 무관하게 일정 — 상승은 막지만 하락도 못 누림
- 초록선(call): K 이하면 시장가 매입(+프리미엄), K 이상이면 K로 cap — **상승 방어 + 하락 누림** (대신 프리미엄 비용)

어떤 게 "좋다"는 답은 회사의 **비대칭 위험 선호**에 달려 있습니다.

## 5. 시나리오 비교 표 — 가격 ±20% 범위에서

In [ ]:
scen_S = [round(S0 * x) for x in [0.80, 0.90, 1.00, 1.10, 1.20]]
rows = []
for s in scen_S:
    spot = s
    fwd = F
    call = min(s, K) + C * np.exp(r*T)
    rows.append({
        'S_T (KRW)': s, 'vs S0': f'{(s/S0 - 1)*100:+.0f}%',
        'unhedged': round(spot, 0),
        'forward': round(fwd, 0),
        'call': round(call, 0),
        'best_strategy': ['unhedged', 'forward', 'call'][np.argmin([spot, fwd, call])],
    })
scen_df = pd.DataFrame(rows)
scen_df

## 6. 변동성에 대한 민감도 — "우리 가격 흔들림이 작아지면 헷지 가치는?"

회사가 공급사 다변화·계약 안정화로 변동성을 낮추면, 콜 프리미엄도 떨어집니다 (=헷징 부담 ↓). 즉 **변동성 관리 자체가 헷징의 일부**입니다.

In [ ]:
sigmas = np.linspace(0.05, max(0.50, sigma_annual * 1.5), 30)
premiums = [bs_call(S0, K, T, r, s) / S0 * 100 for s in sigmas]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sigmas * 100, premiums, color='green')
ax.axvline(sigma_annual * 100, color='black', linewidth=0.6, linestyle=':',
           label=f'현재 σ = {sigma_annual*100:.1f}%')
ax.set_xlabel('연환산 변동성 σ (%)')
ax.set_ylabel('Call 프리미엄 / S0 (%)')
ax.set_title(f'{TARGET} — 변동성에 따른 콜 옵션 비용')
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(DATA_DIR / f'vega_{TARGET}.png', dpi=120)
plt.show()

## 7. 패션 산업의 현실적 적용

거래소 상장 "패션 옵션"은 거의 없지만, 동일한 수학이 다음에 쓰입니다.

| 위험 종류 | 사용 가능한 도구 |
|---|---|
| **환율** (수입 매입) | KRW/USD forward, FX option (시중은행 거래) |
| **원자재** (면, 폴리에스터) | NY ICE Cotton futures, 화학섬유 가격연동 조항 |
| **유가** (운송) | WTI/Brent futures (대형사) 또는 전년도 평균 연동 계약 |
| **공급가 자체** | OTC forward (공급사와 N개월 단가 고정) |
| **수요 위험** | 위탁판매·반품 조항(=공급사가 풋옵션 매도) |

이 노트북의 가격 모델은 **공급사와의 협상 도구**입니다 — "우리 가격 변동성을 본 시장 forward가 얼마인데, 당신 제시가는 더 비싸다"라는 식의 데이터 기반 협상.

## 8. 저장

In [ ]:
out_md = f"""# {TARGET} — Derivatives Hedging (Educational)

- 분석 시점 가격 S0: {S0:,.0f} KRW
- 연환산 변동성 σ: {sigma_annual:.3f}  ({sigma_annual*100:.1f}%)
- 무위험 이자율 r (가정): {r:.3f}
- 만기 T: 6개월

## Forward 가격
{fwd_table.to_markdown(index=False)}

## Call/Put 옵션 가격 (만기 6개월)
{opt_table.to_markdown(index=False)}

## 만기 가격 시나리오별 단위당 비용
{scen_df.to_markdown(index=False)}

## 주의
- 본 분석은 교육 목적이며 투자·헷징 권유가 아닙니다.
- Black-Scholes는 변동성 일정 등 단순 가정에 의존합니다.
- 실제 헷지 설계는 회계 처리(KIFRS 9), 신용·베이시스 위험, 거래 비용 평가가 필요합니다.
"""
out_path = DATA_DIR / f'derivatives_{TARGET}.md'
out_path.write_text(out_md, encoding='utf-8')
print(f'Saved → {out_path.resolve()}')

## 다음 단계

1. `derivatives_*.md`을 Claude Desktop에 첨부 → `derivatives_insight(si)` 프롬프트로 비즈니스 해석.
2. 변동성·만기·행사가 가정을 본인 회사 상황에 맞게 조정 후 재실행.
3. 실제 헷지 설계 전: 회사 재무·법무팀 검토, 거래 상대방 신용도 평가, 회계 처리 영향 분석.